In [2]:
import pandas as pd
import numpy as np
import warnings
import unicodedata
from pathlib import Path

# Configurar la opción para evitar la advertencia
pd.set_option('future.no_silent_downcasting', True)

# Leer el archivo original con TODAS las columnas como texto
archivo = Path('/Users/jhonvilcaranatintaya/Reporte sunedu filtrado.xlsx')
nombre_salida = archivo.with_stem(archivo.stem + "_final")
df = pd.read_excel(archivo, dtype=str)  # ← esta línea es clave

# Reemplazar "-" por NaN y evitar FutureWarning
with warnings.catch_warnings():
    warnings.simplefilter(action='ignore', category=FutureWarning)
    df = df.replace("-", np.nan).infer_objects(copy=False)

# Mapeo explícito de columnas a renombrar
mapeo_columnas = {
    'Filial': 'CODIGO_LOCAL',
    'Semestre': 'PERIODO_ACADEMICO',
    'Facultad': 'CODIGO_FACULTAD_UNIDAD',
    'Escuela': 'CODIGO_PROGRAMA',
    'Cambio Programa': 'CAMBIO_PROGRAMA',
    'Programa Anterior': 'CODIGO_PROGRAMA_ANTERIOR',
    'Fecha de Cambio Programa': 'FECHA_CAMBIO_PROGRAMA',
    'Tipo de Documento': 'TIPO_DOCUMENTO',
    'N° Documento': 'NRO_DOCUMENTO',
    'Código de Estudiante': 'CODIGO_ESTUDIANTE',
    'Ciclo Académico': 'CICLO_ACADEMICO',
    'Modalidad de Estudio': 'MODALIDAD_ESTUDIO',
    'Fecha de Matrícula': 'FECHA_MATRICULA',
    'Tiene periodo anterior': 'TIENE_PERIODO_ANTERIOR',
    'Periodo anterior': 'PERIODO_ACADEMICO_ANTERIOR',
    'N° Créditos Acumulados': 'CREDITOS_ACUMULADOS_APROBADOS_AL_PERIODO_ANTERIOR',
    'Créditos matrículados en el semestre anterior': 'CREDITOS_MATRICULADOS_DEL_PERIODO_ANTERIOR',
    'Créditos aprobados en el semestre anterior': 'CREDITOS_APROBADOS_DEL_PERIODO_ANTERIOR',
    'Nota Promedio Ponderado': 'PROMEDIO_PONDERADO_ACUMULADO_AL_PERIODO_ANTERIOR',
    'Promedio ponderado en el semestre anterior': 'PROMEDIO_PONDERADO_DEL_PERIODO_ANTERIOR',
    'Beca': 'TIENE_BECA',
    'Tipo de Beca': 'TIPO_BECA',
    'Otra Beca': 'OTRA_BECA',
    'Porcentaje Cubierto': 'PORCENTAJE_CUBIERTO'
}

# Lista final de columnas en el orden deseado
columnas_ordenadas = [
    'CODIGO_LOCAL',
    'PERIODO_LECTIVO',
    'TIPO_PERIODO',
    'PERIODO_ACADEMICO',
    'CODIGO_FACULTAD_UNIDAD',
    'CODIGO_PROGRAMA',
    'CAMBIO_PROGRAMA',
    'CODIGO_PROGRAMA_ANTERIOR',
    'FECHA_CAMBIO_PROGRAMA',
    'TIPO_DOCUMENTO',
    'NRO_DOCUMENTO',
    'CODIGO_ESTUDIANTE',
    'CICLO_ACADEMICO',
    'MODALIDAD_ESTUDIO',
    'FECHA_MATRICULA',
    'TIENE_PERIODO_ANTERIOR',
    'PERIODO_ACADEMICO_ANTERIOR',
    'CREDITOS_ACUMULADOS_APROBADOS_AL_PERIODO_ANTERIOR',
    'CREDITOS_MATRICULADOS_DEL_PERIODO_ANTERIOR',
    'CREDITOS_APROBADOS_DEL_PERIODO_ANTERIOR',
    'PROMEDIO_PONDERADO_ACUMULADO_AL_PERIODO_ANTERIOR',
    'PROMEDIO_PONDERADO_DEL_PERIODO_ANTERIOR',
    'CODIGO_ESCALA_PAGO',
    'PORCENTAJE_DESCUENTO',
    'TIENE_BECA',
    'TIPO_BECA',
    'OTRA_BECA',
    'PORCENTAJE_CUBIERTO'
]

# Renombrar columnas que están en el mapeo
df.rename(columns=mapeo_columnas, inplace=True)

# Añadir columnas faltantes como vacías
for col in columnas_ordenadas:
    if col not in df.columns:
        df[col] = ""

# Reordenar columnas
df = df[columnas_ordenadas]


# 1. Asignar 1 a todas las filas en PERIODO_LECTIVO y TIPO_PERIODO
df['PERIODO_LECTIVO'] = 1
df['TIPO_PERIODO'] = 1


# 2. Asignar Valores vacíos  a todas las filas en CAMBIO_PROGRAMA y CODIGO_PROGRAMA_ANTERIOR
df['CAMBIO_PROGRAMA'] = 0
df['CODIGO_PROGRAMA_ANTERIOR'] = ""

# --- LÓGICA CONDICIONAL PARA PERIODO ANTERIOR (SECCIÓN MODIFICADA) ---

# Define las columnas relacionadas con el periodo anterior
columnas_periodo_anterior = [
    'PERIODO_ACADEMICO_ANTERIOR',
    'CREDITOS_ACUMULADOS_APROBADOS_AL_PERIODO_ANTERIOR',
    'CREDITOS_MATRICULADOS_DEL_PERIODO_ANTERIOR',
    'CREDITOS_APROBADOS_DEL_PERIODO_ANTERIOR',
    'PROMEDIO_PONDERADO_ACUMULADO_AL_PERIODO_ANTERIOR',
    'PROMEDIO_PONDERADO_DEL_PERIODO_ANTERIOR'
]

# Asegúrate de que la columna CICLO_ACADEMICO sea un texto limpio
df['CICLO_ACADEMICO'] = df['CICLO_ACADEMICO'].astype(str).str.strip()

# --- REGLA 1: Si el ciclo es '1' ---
# Se crea una "máscara" para identificar a los estudiantes de primer ciclo
mask_ciclo_1 = df["CICLO_ACADEMICO"] == "1"

# Se aplica la regla: no tienen periodo anterior y los campos relacionados se vacían
df.loc[mask_ciclo_1, "TIENE_PERIODO_ANTERIOR"] = 0
df.loc[mask_ciclo_1, columnas_periodo_anterior] = ""

# --- REGLA 2: Si el ciclo es mayor que '1' ---
# Se crea la máscara para los ciclos superiores
mask_ciclos_superiores = df["CICLO_ACADEMICO"] != "1"

# Se aplica la regla: sí tienen periodo anterior
df.loc[mask_ciclos_superiores, "TIENE_PERIODO_ANTERIOR"] = 1

# Ahora, para cada columna de periodo anterior, rellena con '0' SOLO si está vacía
for col in columnas_periodo_anterior:
    # Se identifica qué celdas están vacías o nulas DENTRO de los ciclos superiores
    mask_celda_vacia = df[col].fillna('').astype(str).str.strip() == ''
    
    # Se aplica la regla de rellenar con '0' solo donde ambas condiciones se cumplen
    df.loc[mask_ciclos_superiores & mask_celda_vacia, col] = "0"

# 4. TIENE_BECA: "SI" → 1, "NO" → 0
df['TIENE_BECA'] = df['TIENE_BECA'].replace({'SI': 1, 'NO': 0}).astype('Int64')

# 5. TIPO_BECA:
# - Vacío si TIENE_BECA == 0
# - Mapeo de texto si TIENE_BECA == 1
beca_map = {
    "1": "Beca financiada con fondos públicos nacionales",
    "2": "Beca financiada con fondos de la misma universidad",
    "3": "Beca financiada con otros fondos (nacionales o internacionales)"
}
df.loc[df['TIENE_BECA'] == 0, 'TIPO_BECA'] = None
df.loc[df['TIENE_BECA'] == 1, 'TIPO_BECA'] = df.loc[df['TIENE_BECA'] == 1, 'TIPO_BECA'].astype(str).map(beca_map)

# 6. PORCENTAJE_CUBIERTO: quitar "%" y convertir a entero
df['PORCENTAJE_CUBIERTO'] = df['PORCENTAJE_CUBIERTO'].astype(str).str.replace('%', '').str.strip()
df['PORCENTAJE_CUBIERTO'] = pd.to_numeric(df['PORCENTAJE_CUBIERTO'], errors='coerce').fillna(0).astype(int)

# 6b. Vaciar PORCENTAJE_CUBIERTO si TIENE_BECA == 0
df.loc[df['TIENE_BECA'] == 0, 'PORCENTAJE_CUBIERTO'] = None


# Limpiar los datos (eliminar espacios y normalizar caracteres)
df['CODIGO_PROGRAMA'] = df['CODIGO_PROGRAMA'].apply(lambda x: unicodedata.normalize('NFKD', str(x)).encode('ASCII', 'ignore').decode('ASCII'))

# Asegurarse de que la columna 'CODIGO_PROGRAMA_INGRESO' es de tipo str
df['CODIGO_PROGRAMA'] = df['CODIGO_PROGRAMA'].astype(str)

# Diccionario de reemplazo
reemplazos = {
    #SEGUNDAS ESPECIALIDADES
    "Posgrado de Segunda Especialidad de Ingenieria con Mencion en Ciencia de Datos, A Distancia": 280,
    "Posgrado de Segunda Especialidad de Enfermeria en Gestion en Inmunizaciones": 279, 
    "Posgrado de Segunda Especialidad de Enfermeria en Emergencias y Desastres": 67,
    "Posgrado de Segunda Especialidad de Enfermeria en Gineco Obstetricia": 244,
    "Posgrado de Segunda Especialidad en Enfermeria en Administracion y Gestion": 63,
    "Posgrado de Segunda Especialidad en Enfermeria en Centro Quirurgico": 73,
    "Posgrado de Segunda Especialidad en Enfermeria en Cuidados Intensivos Neonatales": 75,
    "Posgrado de Segunda Especialidad en Enfermeria en Cuidados Intensivos Pediatricos": 76,
    "Posgrado de Segunda Especialidad en Enfermeria en Cuidados Quirurgicos": 77,
    "Posgrado de Segunda Especialidad en Enfermeria en Neurologia y Neurocirugia": 82,
    "Posgrado de Segunda Especialidad en Estadistica Aplicada para Investigacion": 87,
    "Posgrado de Segunda Especialidad de Enfermeria en Oncologia": 83,
    "Posgrado de Segunda Especialidad en Estadistica Aplicada para Investigacion": 87,
    "Posgrado de Segunda Especialidad en Enfermeria en Cuidado Integral Infantil": 74,
    "Posgrado de Segunda Especialidad en Enfermeria en Cuidados Intensivos": 66,
    "Posgrado de Segunda Especialidad en Enfermeria en Cuidados Quirurgicos con mencion en Recuperacion Posanastesica": 220,
    "Posgrado de Segunda Especialidad en Enfermeria en Cuidados Quirurgicos con mencion en Traumatologia y Ortopedia": 79,
    "Posgrado de Segunda Especialidad en Enfermeria en Gestion de Central de Esterilizacion": 80,
    "Posgrado de Segunda Especialidad en Salud Ocupacional": 91,
    "Posgrado de Segunda Especialidad en Enfermeria en Pediatria": 84,
    "Posgrado de Segunda Especialidad en Enfermeria en Cuidados Quirurgicos con mencion en Traumatologia y Ortopedia": 79,
    "Posgrado de Segunda Especialidad en Enfermeria en Cuidados Quirurgicos con mencion en Recuperacion Posanastesica": 220,
    "Posgrado de Segunda Especialidad en Enfermeria en Cuidados Intensivos": 66,
    "Posgrado de Segunda Especialidad en Enfermeria en Cuidado Integral Infantil": 74,
    "Posgrado de Segunda Especialidad en Salud Ocupacional": 91,
    "Posgrado de Segunda Especialidad en Enfermeria en Cardiologia": 64,
    "Posgrado de Segunda Especialidad de Enfermeria en Geriatria": 68,
    "Posgrado de Segunda Especialidad en Enfermeria en Gestion de Central de Esterilizacion": 80,
    "Segunda Especialidad de Enfermeria en Cardiologia": 64,
    "Segunda Especialidad de Enfermeria en Cirugia Cardiovascular": 65,
    "Segunda Especialidad de Enfermeria en Geriatria": 68,
    "Segunda Especialidad de Enfermeria en Nefrologia": 69,
    "Segunda Especialidad de Enfermeria en Pediatria": 84,
    "Segunda Especialidad de Enfermeria en Perfusion Extracorporea y Asistencia Circulatoria": 85,
    "Segunda Especialidad de Enfermeria en Cuidado de la Persona con Diabetes con Mencion en Educacion en Diabetes": 269,
    "Segunda Especialidad en Antiguo Testamento": 71,
    "Segunda Especialidad en Asesoria Genetica en Enfermeria Oncologica": 72,
    "Segunda Especialidad en Enfermeria en Gineco Obstetricia": 81,
    "Segunda Especialidad en Psicologia Clinica y de la Salud": 222,
    "Segunda Especialidad en Estudios Adventistas": 88,
    "Segunda Especialidad en Nuevo Testamento": 89,
    "Segunda Especialidad en Profecias Biblicas": 90,
    "Segunda Especialidad Profesional en Neuropsicologia": 272,
    "Segunda Especialidad Profesional en Psicooncologia y Cuidados Paliativos": 258,
    "Posgrado de Segunda Especialidad en Epidemiologia": 284,
    "Posgrado de Segunda Especialidad en Educacion con Mencion en Ingles como Lengua Extranjera": 285,
    "Segunda Especialidad Profesional en Gestion de Proyectos de Inversion en Nutricion y Desarrollo Social": 287,

    # MAESTRÍAS
    "Posgrado de Maestria BIM Facility Management, A Distancia": 273,
    "Maestria en Administracion de Negocios con mencion en Gestion Empresarial": 246,
    "Posgrado de Maestria en Auditoria Mencion: Auditoria Integral": 40,
    "Maestria en Auditoria con mencion en Auditoria Integral": 40,
    "Posgrado de Maestria en Ciencias de la Familia Mencion: Terapia Familiar": 238,
    "Maestria en Ciencias de la Familia": 238,
    "Maestria en Educacion con mencion en Curriculo": 230,
    "Maestria en Ingenieria Ambiental": 48,
    "Maestria en Ingenieria Ambiental y Desarrollo Sostenible": 247,
    "Maestria en Ingenieria de Sistemas con mencion en Ingenieria de Software": 231,
    "Maestria en Liderazgo Eclesiastico": 50,
    "Maestria en Misionologia": 51,
    "Maestria en Nutricion Humana con Mencion en Alimentacion Basada en Plantas": 270,
    "Maestria en Nutricion Humana con mencion en Nutricion Clinica": 52,
    "Maestria en Nutricion Humana con mencion en Nutricion Vegetariana": 232,
    "Maestria en Teologia": 56,
    "Posgrado de Maestria en Teologia": 56,
    "Maestria en Teologia Biblica": 233,
    "Maestria en Teologia Historica": 234,
    "Posgrado de Maestria en Administracion de Negocios Mencion: Finanzas": 226,
    "Posgrado de Maestria en Administracion de Negocios con mencion en Liderazgo y Gestion Organizacional": 227,
    "Posgrado de Maestria en Administracion de Negocios con mencion en Recursos Humanos": 39,
    "Posgrado de Maestria en Educacion: Administracion Educativa": 42,
    "Posgrado de Maestria en Educacion: Investigacion y Docencia Universitaria": 228,
    "Posgrado de Maestria en Educacion: Mencion en Psicologia Educativa": 229,
    "Posgrado de Maestria en Enfermeria con Mencion en Administracion y Gestion": 46,
    "Posgrado de Maestria en Ingenieria de Sistemas con mencion en Direccion y Gestion de Tecnologias de Informacion": 49,
    "Posgrado de Maestria en Psicologia Clinica y de la Salud": 53,
    "Posgrado de Maestria en Salud Publica Mencion: Gestion de los Servicios de Salud": 223,
    "Posgrado de Maestria en Salud Publica Mencion: Salud Colectiva y Promocion de la Salud": 224,
    "Posgrado de Maestria en Terapia Familiar y de Pareja": 41,
    "Posgrado de Maestria en Administracion de Negocios Mencion: Administracion de Recursos Humanos": 39,
    "Posgrado de Maestria en Administracion de Negocios Mencion: Gestion Empresarial ": 246,
    "Posgrado de Maestria en Ciencias de la Familia Mencion: Terapia Familiar": 41,
    "Posgrado de Maestria en Ingenieria Ambiental": 48,
    "Posgrado de Maestria en Ingenieria de Sistemas : Direccion y Gestion de TI": 49,
    "Posgrado de Maestria en Nutricion Humana con Mencion en Nutricion Vegetariana": 232,
    "Posgrado de Maestria en Teologia Mencion: Liderazgo y Administracion Eclesiastica": 56,
    "Posgrado de Maestria en Misionologia": 51,
    "Posgrado de Maestria en Auditoria Mencion: Auditoria Integral": 40,
    "Posgrado de Maestria en Nutricion Humana con Mencion en Nutricion Vegetariana": 232,
    "Posgrado de Maestria en Teologia Biblica": 233,
    "Posgrado de Maestria en Administracion de Negocios Mencion: Gestion Empresarial": 246,

    # DOCTORADOS
    "Doctorado en Administracion de Negocios": 11,
    "Posgrado de Doctorado en Administracion de Negocios": 11,
    "Doctorado en Ciencias Contables y Financieras": 276,
    "Posgrado de Doctorado en Educacion mencion: Gestion Educativa": 235,
    "Doctorado en Educacion con mencion en Curriculo y Docencia": 12,
    "Posgrado de Doctorado en Enfermeria": 13,
    "Doctorado en Ingenieria de Sistemas": 14,
    "Posgrado de Doctorado en Salud Publica": 15,
    "Doctorado en Teologia": 16,
    "Posgrado de Doctorado en Teologia": 16,

    
}

# Realizar el reemplazo en la columna 'CODIGO_PROGRAMA_INGRESO'
df['CODIGO_PROGRAMA'] = df['CODIGO_PROGRAMA'].replace(reemplazos)

# Diccionario de reemplazo para TIPO_DOCUMENTO
tipo_documento_map = {
    "DNI": 1,
    "PASAPORTE": 2,
    "CE": 3,
    "CÉDULA DE IDENTIDAD": 4,
    "DIDEXT": 5,
    "DIDExt": 5,
    "PERMISO TEMPORAL DE PERMANENCIA": 6,
    "CARNÉ DE IDENTIDAD": 7,
    "CÉDULA DE CIUDADANÍA": 8,
    "CARNÉ TEMPORAL DE PERMANENCIA": 9
}

# Aplicar el reemplazo en TIPO_DOCUMENTO
df['TIPO_DOCUMENTO'] = df['TIPO_DOCUMENTO'].replace(tipo_documento_map)



# Reemplazar todos los valores de CODIGO_LOCAL con 'SL01'
df['CODIGO_LOCAL'] = 'SL01'

# Diccionario de facultad a código
facultad_unidad_map = {
    "Ciencias de la Salud": 7,
    "Ciencias Empresariales": 6,
    "Ingeniería y Arquitectura": 3,
    "Escuela General de Posgrado": 9,
    "No Consignado": 2,
    "Ciencias Humanas y Educación": 5,
    "Teología": 1
}

# Reemplazar CODIGO_FACULTAD_UNIDAD según el diccionario
df['CODIGO_FACULTAD_UNIDAD'] = df['CODIGO_FACULTAD_UNIDAD'].replace(facultad_unidad_map)

# Diccionario de modalidad de estudio
reemplazos_modalidad = {
    "Dato no declarado": 0,
    "Presencial": 1,
    "Semipresencial": 2,
    "A Distancia": 3
}

# Reemplazar MODALIDAD_ESTUDIO según el diccionario
df['MODALIDAD_ESTUDIO'] = df['MODALIDAD_ESTUDIO'].replace(reemplazos_modalidad)


cod = pd.to_numeric(df["CODIGO_PROGRAMA"], errors="coerce")
df.loc[cod.eq(222), "CODIGO_FACULTAD_UNIDAD"] = 7

# VALIDAR SOLO SI EXISTE ALGÚN DOCUMENTO INVÁLIDO
reglas_documento_motivo = {
    1: lambda x: ("DNI debe tener exactamente 8 dígitos" if not (str(x).isdigit() and len(x) == 8) else None),
    2: lambda x: ("Pasaporte debe tener mínimo 6 caracteres" if not (len(str(x).strip()) >= 6) else None),
    3: lambda x: ("CE debe tener al menos 8 dígitos numéricos" if not (str(x).isdigit() and len(x) >= 8) else None),
    4: lambda x: ("Cédula de Identidad debe tener mínimo 6 caracteres" if not (len(str(x).strip()) >= 6) else None),
    5: lambda x: ("DIDEXT debe tener mínimo 6 caracteres" if not (len(str(x).strip()) >= 6) else None),
    6: lambda x: ("PTP debe tener mínimo 6 caracteres" if not (len(str(x).strip()) >= 6) else None),
    7: lambda x: ("Carné de Identidad debe tener mínimo 6 caracteres" if not (len(str(x).strip()) >= 6) else None),
    8: lambda x: ("Cédula de Ciudadanía debe tener mínimo 6 caracteres" if not (len(str(x).strip()) >= 6) else None),
    9: lambda x: ("Carné Temporal de Permanencia debe tener mínimo 6 caracteres" if not (len(str(x).strip()) >= 6) else None)
}
# Aplicar las reglas fila por fila
def validar_documento_con_motivo(row):
    tipo = row["TIPO_DOCUMENTO"]
    nro = row["NRO_DOCUMENTO"]
    try:
        validador = reglas_documento_motivo.get(int(tipo), lambda x: None)
        return validador(nro)
    except:
        return " Tipo de documento inválido o error inesperado"


# Crear un dataframe solo con filas inválidas (sin modificar df original)
df_doc_obs = df.copy()
df_doc_obs["MOTIVO"] = df_doc_obs.apply(validar_documento_con_motivo, axis=1)
df_doc_obs = df_doc_obs[df_doc_obs["MOTIVO"].notnull()]

# Mostrar resumen si existen observaciones
if not df_doc_obs.empty:
    print(f"\n ❌ Se encontraron {len(df_doc_obs)} documentos inválidos:")
    display(df_doc_obs[["TIPO_DOCUMENTO", "NRO_DOCUMENTO", "MOTIVO"]])

df['FECHA_MATRICULA'] = ""

# Guardar el resultado en un nuevo archivo Excel
df.to_excel(nombre_salida, index=False)

print("\n⚠️ Colocar la FECHA_MATRICULA de acuerdo al SIU")
print(f"\n✅ Archivo guardado como '{nombre_salida}'.")



 ❌ Se encontraron 1 documentos inválidos:


,TIPO_DOCUMENTO,NRO_DOCUMENTO,MOTIVO
47,3,CarEx : 0604739029,CE debe tener al menos 8 dígitos numéricos



⚠️ Colocar la FECHA_MATRICULA de acuerdo al SIU

✅ Archivo guardado como '/Users/jhonvilcaranatintaya/Reporte sunedu filtrado_final.xlsx'.


In [ ]:
Recordar que 81 y 244, uno es primer ciclo y el otro segundo ciclo respectivamente, 
ya que en el registro el primero no tiene tildes que indica que es de 1er ciclo
81 es facultad 9
244 es no consignado 2

si CODIGO_PROGRAMA_OPCIONES y CODIGO_PROGRAMA_INGRESO tienen valor de 81 entonces CODIGO_FACULTAD_UNIDAD Y CODIGO_FACULTAD_UNIDAD_INGRESO es 9
si CODIGO_PROGRAMA_OPCIONES es 81 entonces CODIGO_FACULTAD_UNIDAD es 9 también si CODIGO_PROGRAMA_INGRESO es 81 entonces CODIGO_FACULTAD_UNIDAD_INGRESO es 9


222 Segunda Especialidad en Psicología Clínica y de la Salud es 7 Ciencias de la salud
si CODIGO_PROGRAMA_OPCIONES es 222 entonces CODIGO_FACULTAD_UNIDAD es 7 también si CODIGO_PROGRAMA_INGRESO es 222 entonces CODIGO_FACULTAD_UNIDAD_INGRESO es 7

Hay 2 cuidados intensivos